# T02 — Exploratory data analysis and empirical characterization

## Purpose and inherited constraints

This notebook characterizes the validated, manifest-selected T01 dataset before modeling. It is descriptive: it does not redefine the ITT CATE estimand, select features, assign business meanings to `f0`–`f11`, train models, construct the outer split, or access held-out evaluation. `X` remains exactly `f0`–`f11`, `T=treatment`, and primary `Y=conversion`; `visit` is excluded from primary T02 outputs and `exposure` remains audit-only.

T02 reports missingness only. Model-facing missing-value handling is deferred to T04 and estimator-specific contracts. Support statements here apply only to the current global or univariate descriptive population; future folds, top-K groups, segments, or leaves must apply their own downstream gates when those populations exist.

## Accepted T02 decisions and execution map

- **T02-D01 — FROZEN_INHERITED:** EDA cannot establish causal effects or change the estimand.
- **T02-D02 — IMPLEMENTATION_ONLY:** retain only question-led tables and figures.
- **T02-D03 — IMPLEMENTATION_ONLY:** authoritative outputs are immutable and run-scoped.

Execution: checksum-aware loading → invariant reconciliation → `(T,Y)` summaries → per-feature cardinality/quantiles → overall versus arm-stratified descriptions → bounded support findings → purposeful figures → evidence-bounded interpretation → immutable manifest closure.

In [1]:
from __future__ import annotations

import copy
import gc
import hashlib
import io
import json
import platform
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.compute as pc
from IPython.display import Markdown, display

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY if (WORKING_DIRECTORY / "src").is_dir() else WORKING_DIRECTORY.parent
if not (REPO_ROOT / "src" / "data.py").is_file():
    raise RuntimeError(f"Cannot locate repository root from {WORKING_DIRECTORY}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    AUDIT_ONLY_COLUMN,
    EXPECTED_ARROW_TYPES,
    FEATURE_COLUMNS,
    PRIMARY_OUTCOME,
    PROCESSED_COLUMNS,
    PROCESSED_SCHEMA,
    SECONDARY_OUTCOME,
    SOURCE_ROW_ID,
    TREATMENT_COLUMN,
    assert_model_feature_contract,
    finalize_artifact_manifest,
    implementation_environment,
    load_selector,
    materialize_pandas,
    open_processed_dataset,
    portable_repo_path,
    scan_batches,
    sha256_file,
    write_bytes_new,
    write_json_new,
    write_text_new,
)

NOTEBOOK_PATH = REPO_ROOT / "notebooks" / "03_exploratory_data_analysis.ipynb"
SELECTOR_PATH = REPO_ROOT / "configs" / "data_manifest.json"
STAGE = "pre_split_descriptive_eda"
POPULATION = "validated_unsplit_released_rows"
QUANTILES = (0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99)
QUANTILE_COLUMNS = {q: f"q{int(q * 100):02d}" for q in QUANTILES}

assert NOTEBOOK_PATH.is_file(), NOTEBOOK_PATH
assert_selector_columns = tuple(FEATURE_COLUMNS)
assert_model_feature_contract(assert_selector_columns)

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def notebook_source_sha256(path: Path) -> str:
    payload = json.loads(path.read_text(encoding="utf-8"))
    sources = [
        {"cell_type": cell["cell_type"], "source": "".join(cell.get("source", []))}
        for cell in payload["cells"]
    ]
    canonical = json.dumps(sources, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def write_csv_new(frame: pd.DataFrame, run_root: Path, relative_path: str) -> Path:
    payload = frame.to_csv(index=False, lineterminator="\n")
    return write_text_new(run_root, relative_path, payload)

def save_figure_new(fig, run_root: Path, relative_path: str, *, metadata: dict[str, str]) -> Path:
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", metadata=metadata)
    return write_bytes_new(run_root, relative_path, buffer.getvalue())

def finite_or_none(value):
    if value is None or pd.isna(value) or not np.isfinite(value):
        return None
    return float(value)

def format_optional(value) -> str:
    return "NA" if value is None or pd.isna(value) else f"{float(value):.8g}"


## 1. T01 manifest/checksum-aware loading

**Question:** Is this run consuming the exact T01 processed artifact selected by an explicit manifest and verified checksum?

In [2]:
RUN_CREATED_AT = utc_now()
RUN_ID = datetime.now(timezone.utc).strftime("t02_eda_%Y%m%dT%H%M%SZ_%f")
RUN_ROOT = REPO_ROOT / "outputs" / "runs" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)

selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)
processed_sha256 = selector.payload["processed_sha256"]
notebook_source_hash = notebook_source_sha256(NOTEBOOK_PATH)
src_data_hash = sha256_file(REPO_ROOT / "src" / "data.py")
try:
    git_head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    git_dirty = bool(subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_ROOT, check=True,
        capture_output=True, text=True,
    ).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    git_head, git_dirty = None, None

run_config = {
    "run_id": RUN_ID, "task": "T02", "issue": 3,
    "stage": STAGE, "population": POPULATION,
    "input_selector": "configs/data_manifest.json",
    "feature_columns": list(FEATURE_COLUMNS),
    "treatment": TREATMENT_COLUMN, "primary_outcome": PRIMARY_OUTCOME,
    "visit_in_primary_outputs": False, "exposure_role": "audit_only",
    "quantiles": list(QUANTILES), "quantile_interpolation": "linear",
    "outer_split_constructed": False, "held_out_access": False,
    "model_training": False, "feature_selection": False,
    "missing_value_action": "report_only_model_handling_deferred_to_T04",
    "code_identity": {
        "notebook_sources_sha256": notebook_source_hash,
        "src/data.py_sha256": src_data_hash,
        "git_head": git_head, "git_dirty": git_dirty,
    },
}
write_json_new(RUN_ROOT, "audit/run_config.json", run_config)

environment = implementation_environment()
environment.update({
    "run_id": RUN_ID, "created_at_utc": RUN_CREATED_AT,
    "matplotlib": matplotlib.__version__, "nbformat": nbformat.__version__,
    "notebook_sources_sha256": notebook_source_hash,
    "git_head": git_head, "git_dirty": git_dirty,
})
write_json_new(RUN_ROOT, "audit/environment.json", environment)

data_manifest = copy.deepcopy(selector.payload)
data_manifest["manifest_role"] = "IMMUTABLE_RUN_SNAPSHOT"
data_manifest["run_id"] = RUN_ID
data_manifest["resolved_at_utc"] = RUN_CREATED_AT
data_manifest["selector_path"] = portable_repo_path(SELECTOR_PATH, REPO_ROOT)
data_manifest["raw_path"] = portable_repo_path(selector.raw_csv_path, REPO_ROOT)
data_manifest["raw_compressed_path"] = portable_repo_path(selector.raw_compressed_path, REPO_ROOT)
data_manifest["processed_path"] = portable_repo_path(selector.processed_path, REPO_ROOT)
data_manifest["processed_size_bytes"] = selector.processed_path.stat().st_size
write_json_new(RUN_ROOT, "audit/data_manifest.json", data_manifest)

display(pd.DataFrame([{
    "run_id": RUN_ID, "dataset": selector.payload["dataset_name"],
    "processed_path": data_manifest["processed_path"],
    "processed_sha256": processed_sha256, "population": POPULATION,
}]))

,run_id,dataset,processed_path,processed_sha256,population
0,t02_eda_20260813T094750Z_546902,CRITEO-UPLIFTv2.1,data/processed/criteo-uplift-v2.1.parquet,fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c...,validated_unsplit_released_rows


## 2. Frozen-invariant reconciliation

**Question:** Does the selected artifact still satisfy the frozen schema, roles, precision, label, and source-row identity contracts? Missingness is characterized only; no model-facing handling is selected here.

In [3]:
scan_started = time.perf_counter()
process = psutil.Process()
peak_rss_bytes = process.memory_info().rss
row_count = 0
source_order_ok = True
feature_missing = {feature: 0 for feature in FEATURE_COLUMNS}
feature_nonfinite = {feature: 0 for feature in FEATURE_COLUMNS}
arm_counts = np.zeros(2, dtype=np.int64)
outcome_counts = np.zeros(2, dtype=np.int64)
joint_counts = np.zeros(4, dtype=np.int64)

scan_columns = tuple(FEATURE_COLUMNS) + (TREATMENT_COLUMN, PRIMARY_OUTCOME, SOURCE_ROW_ID)
for batch in scan_batches(dataset, columns=scan_columns):
    n_batch = batch.num_rows
    for feature in FEATURE_COLUMNS:
        values = batch.column(batch.schema.get_field_index(feature))
        feature_missing[feature] += int(pc.sum(pc.is_null(values)).as_py() or 0)
        feature_nonfinite[feature] += int(pc.sum(pc.invert(pc.is_finite(values))).as_py() or 0)
    t = batch.column(batch.schema.get_field_index(TREATMENT_COLUMN)).to_numpy(zero_copy_only=False)
    y = batch.column(batch.schema.get_field_index(PRIMARY_OUTCOME)).to_numpy(zero_copy_only=False)
    ids = batch.column(batch.schema.get_field_index(SOURCE_ROW_ID)).to_numpy(zero_copy_only=False)
    if not np.isin(t, (0, 1)).all() or not np.isin(y, (0, 1)).all():
        raise AssertionError("T and Y must be complete binary labels")
    expected_ids = np.arange(row_count, row_count + n_batch, dtype=np.int64)
    source_order_ok = source_order_ok and np.array_equal(ids, expected_ids)
    arm_counts += np.bincount(t, minlength=2)[:2]
    outcome_counts += np.bincount(y, minlength=2)[:2]
    joint_counts += np.bincount(t * 2 + y, minlength=4)[:4]
    row_count += n_batch
    peak_rss_bytes = max(peak_rss_bytes, process.memory_info().rss)

invariant_scan_elapsed_seconds = time.perf_counter() - scan_started
schema_ok = dataset.schema.remove_metadata().equals(PROCESSED_SCHEMA, check_metadata=False)
expected_rows = int(selector.payload["expected_rows"])
feature_float64_ok = all(dataset.schema.field(f).type == pa.float64() for f in FEATURE_COLUMNS)
columns_ok = tuple(dataset.schema.names) == PROCESSED_COLUMNS
arm_support_ok = bool((arm_counts > 0).all())
source_identity_ok = source_order_ok and row_count == expected_rows

invariant_rows = [
    ("HG-01", "processed checksum pinned and verified before open", processed_sha256, "64-character SHA-256", True),
    ("HG-02", "row count", row_count, expected_rows, row_count == expected_rows),
    ("HG-03", "processed schema and canonical column order", list(dataset.schema.names), list(PROCESSED_COLUMNS), schema_ok and columns_ok),
    ("HG-03-X", "canonical model feature order", list(FEATURE_COLUMNS), list(FEATURE_COLUMNS), True),
    ("HG-04", "feature analytical precision", "float64", "float64", feature_float64_ok),
    ("HG-05", "feature finite values", int(sum(feature_nonfinite.values())), 0, sum(feature_nonfinite.values()) == 0),
    ("HG-06", "global observed assignment-arm support", arm_counts.tolist(), "both arms non-zero", arm_support_ok),
    ("HG-07", "source-row ordinals complete, unique, and canonical-order preserving", source_identity_ok, True, source_identity_ok),
]
data_summary = pd.DataFrame(invariant_rows, columns=["check_id", "question", "observed_value", "expected_value", "passed"])
data_summary["run_id"] = RUN_ID
data_summary["population"] = POPULATION
data_summary["evidence_type"] = "HARD_GATE"
data_summary["evidence_status"] = np.where(data_summary["passed"], "PASS", "FAIL")
data_summary["required_action"] = np.where(data_summary["passed"], "PASS", "STOP")
missing_rows = pd.DataFrame({
    "check_id": [f"ED-01-{f}" for f in FEATURE_COLUMNS],
    "question": [f"Characterize observed missingness for {f}; PASS applies only to this diagnostic" for f in FEATURE_COLUMNS],
    "observed_value": [feature_missing[f] for f in FEATURE_COLUMNS],
    "expected_value": ["diagnostic characterization only; model-facing handling deferred to T04"] * len(FEATURE_COLUMNS),
    "passed": [True] * len(FEATURE_COLUMNS),
    "run_id": RUN_ID, "population": POPULATION,
    "evidence_type": "EMPIRICAL_DIAGNOSTIC",
    "evidence_status": "INFO", "required_action": "PASS",
})
data_summary = pd.concat([data_summary, missing_rows], ignore_index=True)
data_summary = data_summary[["run_id", "population", "check_id", "evidence_type", "question", "observed_value", "expected_value", "evidence_status", "required_action", "passed"]]
write_csv_new(data_summary, RUN_ROOT, "tables/data_summary.csv")
if not bool(data_summary.loc[data_summary["evidence_type"] == "HARD_GATE", "passed"].all()):
    raise AssertionError("T02 input invariant reconciliation failed")
display(Markdown("**Missingness boundary:** ED-01 `PASS` means only that the diagnostic was completed and the observed condition was recorded. Model-facing missing-value handling remains deferred to T04."))
display(data_summary)

**Missingness boundary:** ED-01 `PASS` means only that the diagnostic was completed and the observed condition was recorded. Model-facing missing-value handling remains deferred to T04.

,run_id,population,check_id,evidence_type,question,observed_value,expected_value,evidence_status,required_action,passed
0,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-01,HARD_GATE,processed checksum pinned and verified before ...,fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c...,64-character SHA-256,PASS,PASS,True
1,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-02,HARD_GATE,row count,13979592,13979592,PASS,PASS,True
2,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-03,HARD_GATE,processed schema and canonical column order,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...","[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...",PASS,PASS,True
3,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-03-X,HARD_GATE,canonical model feature order,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...","[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...",PASS,PASS,True
4,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-04,HARD_GATE,feature analytical precision,float64,float64,PASS,PASS,True
5,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-05,HARD_GATE,feature finite values,0,0,PASS,PASS,True
6,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-06,HARD_GATE,global observed assignment-arm support,"[2096937, 11882655]",both arms non-zero,PASS,PASS,True
7,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,HG-07,HARD_GATE,"source-row ordinals complete, unique, and cano...",True,True,PASS,PASS,True
8,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,ED-01-f0,EMPIRICAL_DIAGNOSTIC,Characterize observed missingness for f0; PASS...,0,diagnostic characterization only; model-facing...,INFO,PASS,True
9,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,ED-01-f1,EMPIRICAL_DIAGNOSTIC,Characterize observed missingness for f1; PASS...,0,diagnostic characterization only; model-facing...,INFO,PASS,True


## 3. Treatment, outcome, and joint `(T,Y)` cells

**Question:** How do observed assignment imbalance and conversion rarity differ, and are all four global theoretical cells represented? These are descriptive counts, not proof of randomization or a causal effect.

In [4]:
full_cells = pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME])
treatment_outcome_summary = pd.DataFrame({
    TREATMENT_COLUMN: [0, 0, 1, 1],
    PRIMARY_OUTCOME: [0, 1, 0, 1],
    "n": joint_counts.tolist(),
})
assert list(treatment_outcome_summary.set_index([TREATMENT_COLUMN, PRIMARY_OUTCOME]).index) == list(full_cells)
treatment_outcome_summary["arm_n"] = treatment_outcome_summary.groupby(TREATMENT_COLUMN)["n"].transform("sum")
treatment_outcome_summary["joint_fraction"] = treatment_outcome_summary["n"] / row_count
treatment_outcome_summary["within_arm_fraction"] = treatment_outcome_summary["n"] / treatment_outcome_summary["arm_n"]
treatment_outcome_summary["treatment_fraction"] = treatment_outcome_summary["arm_n"] / row_count
treatment_outcome_summary["evidence_status"] = np.where(treatment_outcome_summary["n"] == 0, "WARNING", "INFO")
treatment_outcome_summary["required_action"] = np.where(treatment_outcome_summary["n"] == 0, "WARNING", "PASS")
treatment_outcome_summary["run_id"] = RUN_ID
treatment_outcome_summary["population"] = POPULATION
treatment_outcome_summary = treatment_outcome_summary[["run_id", "population", TREATMENT_COLUMN, PRIMARY_OUTCOME, "n", "arm_n", "joint_fraction", "within_arm_fraction", "treatment_fraction", "evidence_status", "required_action"]]
assert int(treatment_outcome_summary["n"].sum()) == row_count
assert np.allclose(treatment_outcome_summary.groupby(TREATMENT_COLUMN)["within_arm_fraction"].sum(), 1.0)
write_csv_new(treatment_outcome_summary, RUN_ROOT, "tables/treatment_outcome_summary.csv")
display(treatment_outcome_summary)

,run_id,population,treatment,conversion,n,arm_n,joint_fraction,within_arm_fraction,treatment_fraction,evidence_status,required_action
0,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,0,0,2092874,2096937,0.149709,0.998062,0.15,INFO,PASS
1,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,0,1,4063,2096937,0.000291,0.001938,0.15,INFO,PASS
2,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,1,0,11845944,11882655,0.847374,0.996911,0.85,INFO,PASS
3,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,1,1,36711,11882655,0.002626,0.003089,0.85,INFO,PASS


## 4. Feature cardinality, quantiles, and overall versus arm-stratified distributions

**Question:** What center, spread, tails, cardinality, mass concentration, and arm-specific observed support characterize each anonymous feature? Each operation materializes only one feature plus `T`; no feature is selected, removed, imputed, clipped, or assigned semantics. Range comparison is explicitly a **univariate observed-range overlap diagnostic**, not proof of multivariate overlap or positivity.

In [5]:
feature_rows, arm_rows, top_value_rows, feature_resource_rows = [], [], [], []

def describe_series(series: pd.Series) -> dict:
    quantile_values = series.quantile(QUANTILES, interpolation="linear")
    result = {
        "n": int(series.size), "non_null_n": int(series.notna().sum()),
        "missing_n": int(series.isna().sum()), "missing_fraction": float(series.isna().mean()),
        "mean": finite_or_none(series.mean()), "std": finite_or_none(series.std(ddof=1)),
        "min": finite_or_none(series.min()), "max": finite_or_none(series.max()),
    }
    result.update({QUANTILE_COLUMNS[q]: finite_or_none(quantile_values.loc[q]) for q in QUANTILES})
    return result

for feature in FEATURE_COLUMNS:
    started = time.perf_counter()
    frame = materialize_pandas(dataset, columns=[feature, TREATMENT_COLUMN], row_limit=None)
    if str(frame[feature].dtype) != "float64":
        raise AssertionError(f"{feature} lost float64 precision")
    overall = describe_series(frame[feature])
    overall.update({
        "run_id": RUN_ID, "population": POPULATION, "feature": feature,
        "n_unique_non_null": int(frame[feature].nunique(dropna=True)),
        "unique_fraction_non_null": (float(frame[feature].nunique(dropna=True) / overall["non_null_n"]) if overall["non_null_n"] else None),
    })
    feature_rows.append(overall)

    value_counts = frame[feature].value_counts(dropna=False).head(5)
    for rank, (value, count) in enumerate(value_counts.items(), start=1):
        top_value_rows.append({
            "run_id": RUN_ID, "population": POPULATION, "feature": feature,
            "rank": rank, "value": "<MISSING>" if pd.isna(value) else repr(float(value)),
            "n": int(count), "fraction": float(count / row_count),
        })

    arm_descriptions = {}
    for arm in (0, 1):
        arm_series = frame.loc[frame[TREATMENT_COLUMN] == arm, feature]
        description = describe_series(arm_series)
        arm_descriptions[arm] = description
        description.update({
            "run_id": RUN_ID, "population": POPULATION,
            "feature": feature, TREATMENT_COLUMN: arm,
            "n_unique_non_null": int(arm_series.nunique(dropna=True)),
        })
        arm_rows.append(description)

    lower = max(arm_descriptions[0]["min"], arm_descriptions[1]["min"]) if arm_descriptions[0]["min"] is not None and arm_descriptions[1]["min"] is not None else None
    upper = min(arm_descriptions[0]["max"], arm_descriptions[1]["max"]) if arm_descriptions[0]["max"] is not None and arm_descriptions[1]["max"] is not None else None
    overlap_exists = bool(lower is not None and upper is not None and lower <= upper)
    for row in arm_rows[-2:]:
        row["univariate_observed_range_overlap_diagnostic"] = overlap_exists
        row["univariate_observed_range_overlap_lower"] = lower
        row["univariate_observed_range_overlap_upper"] = upper
        row["univariate_observed_range_overlap_limitation"] = "not proof of multivariate overlap or positivity"

    del frame, value_counts
    gc.collect()
    peak_rss_bytes = max(peak_rss_bytes, process.memory_info().rss)
    feature_resource_rows.append({
        "feature": feature, "elapsed_seconds": time.perf_counter() - started,
        "rss_bytes_after_cleanup": process.memory_info().rss,
    })

feature_order = {feature: index for index, feature in enumerate(FEATURE_COLUMNS)}
feature_summary = pd.DataFrame(feature_rows)
feature_summary["_feature_order"] = feature_summary["feature"].map(feature_order)
feature_summary = feature_summary.sort_values("_feature_order").drop(columns="_feature_order").reset_index(drop=True)
feature_arm_summary = pd.DataFrame(arm_rows)
feature_arm_summary["_feature_order"] = feature_arm_summary["feature"].map(feature_order)
feature_arm_summary = feature_arm_summary.sort_values(["_feature_order", TREATMENT_COLUMN]).drop(columns="_feature_order").reset_index(drop=True)
feature_top_values = pd.DataFrame(top_value_rows)
feature_top_values["_feature_order"] = feature_top_values["feature"].map(feature_order)
feature_top_values = feature_top_values.sort_values(["_feature_order", "rank"]).drop(columns="_feature_order").reset_index(drop=True)

for frame_to_check in (feature_summary, feature_arm_summary):
    quantile_matrix = frame_to_check[[QUANTILE_COLUMNS[q] for q in QUANTILES]].to_numpy(dtype=float)
    for quantile_row in quantile_matrix:
        finite_quantiles = quantile_row[np.isfinite(quantile_row)]
        if finite_quantiles.size > 1 and not np.all(np.diff(finite_quantiles) >= 0):
            raise AssertionError("Finite quantiles must be monotonic")
assert feature_summary["feature"].tolist() == list(FEATURE_COLUMNS)
assert feature_arm_summary.groupby("feature").size().eq(2).all()
assert feature_summary["n"].eq(row_count).all()

write_csv_new(feature_summary, RUN_ROOT, "tables/feature_summary.csv")
write_csv_new(feature_arm_summary, RUN_ROOT, "tables/feature_arm_summary.csv")
write_csv_new(feature_top_values, RUN_ROOT, "tables/feature_top_values.csv")
display(feature_summary)
display(feature_arm_summary)

,n,non_null_n,missing_n,missing_fraction,mean,std,min,max,q01,q05,q25,q50,q75,q95,q99,run_id,population,feature,n_unique_non_null,unique_fraction_non_null
0,13979592,13979592,0,0.0,19.620297,5.377464,12.616365,26.745255,12.616365,12.616365,12.616365,21.923413,24.436459,26.311866,26.672876,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f0,2181959,0.156082
1,13979592,13979592,0,0.0,10.069977,0.104756,10.059654,16.344187,10.059654,10.059654,10.059654,10.059654,10.059654,10.059654,10.679513,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f1,60,0.000004
2,13979592,13979592,0,0.0,8.446582,0.299316,8.214383,9.051962,8.214383,8.214383,8.214383,8.214383,8.723335,9.004230,9.042916,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f2,2051900,0.146778
3,13979592,13979592,0,0.0,4.178923,1.336645,-8.398387,4.679882,-1.733228,0.973841,4.679882,4.679882,4.679882,4.679882,4.679882,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f3,552,0.000039
4,13979592,13979592,0,0.0,10.338837,0.343308,10.280525,21.123508,10.280525,10.280525,10.280525,10.280525,10.280525,10.280525,11.973287,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f4,260,0.000019
5,13979592,13979592,0,0.0,4.028513,0.431097,-9.011892,4.115453,2.230907,3.013064,4.115453,4.115453,4.115453,4.115453,4.115453,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f5,132,0.000009
6,13979592,13979592,0,0.0,-4.155356,4.577914,-31.429784,0.294443,-17.777273,-13.353455,-6.699321,-2.411115,0.294443,0.294443,0.294443,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f6,1645,0.000118
7,13979592,13979592,0,0.0,5.101765,1.205248,4.833815,11.998401,4.833815,4.833815,4.833815,4.833815,4.833815,6.045297,11.482084,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f7,622143,0.044504
8,13979592,13979592,0,0.0,3.933581,0.056660,3.635107,3.971858,3.751603,3.806309,3.910792,3.971858,3.971858,3.971858,3.971858,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f8,3743,0.000268
9,13979592,13979592,0,0.0,16.027638,7.018975,13.190056,75.295017,13.190056,13.190056,13.190056,13.190056,13.190056,33.712556,44.893640,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f9,1594,0.000114


,n,non_null_n,missing_n,missing_fraction,mean,std,min,max,q01,q05,...,q99,run_id,population,feature,treatment,n_unique_non_null,univariate_observed_range_overlap_diagnostic,univariate_observed_range_overlap_lower,univariate_observed_range_overlap_upper,univariate_observed_range_overlap_limitation
0,2096937,2096937,0,0.0,19.651705,5.388112,12.616365,26.745255,12.616365,12.616365,...,26.673678,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f0,0,1024332,True,12.616365,26.745255,not proof of multivariate overlap or positivity
1,11882655,11882655,0,0.0,19.614755,5.375564,12.616365,26.745255,12.616365,12.616365,...,26.672737,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f0,1,2120723,True,12.616365,26.745255,not proof of multivariate overlap or positivity
2,2096937,2096937,0,0.0,10.067935,0.092990,10.059654,15.600396,10.059654,10.059654,...,10.679513,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f1,0,36,True,10.059654,15.600396,not proof of multivariate overlap or positivity
3,11882655,11882655,0,0.0,10.070337,0.106693,10.059654,16.344187,10.059654,10.059654,...,10.679513,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f1,1,58,True,10.059654,15.600396,not proof of multivariate overlap or positivity
4,2096937,2096937,0,0.0,8.448173,0.300676,8.214383,9.051962,8.214383,8.214383,...,9.043058,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f2,0,758769,True,8.214383,9.051962,not proof of multivariate overlap or positivity
5,11882655,11882655,0,0.0,8.446302,0.299075,8.214383,9.051962,8.214383,8.214383,...,9.042888,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f2,1,1961427,True,8.214383,9.051962,not proof of multivariate overlap or positivity
6,2096937,2096937,0,0.0,4.232821,1.242029,-8.398387,4.679882,-1.450558,1.267425,...,4.679882,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f3,0,398,True,-8.376438,4.679882,not proof of multivariate overlap or positivity
7,11882655,11882655,0,0.0,4.169412,1.352432,-8.376438,4.679882,-1.765785,0.842442,...,4.679882,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f3,1,544,True,-8.376438,4.679882,not proof of multivariate overlap or positivity
8,2096937,2096937,0,0.0,10.336526,0.338681,10.280525,21.123508,10.280525,10.280525,...,11.973287,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f4,0,162,True,10.280525,20.366604,not proof of multivariate overlap or positivity
9,11882655,11882655,0,0.0,10.339245,0.344117,10.280525,20.366604,10.280525,10.280525,...,11.973287,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,f4,1,248,True,10.280525,20.366604,not proof of multivariate overlap or positivity


## 5. Bounded support findings and downstream implications

**Question:** Which current global or univariate observations should downstream tasks carry forward? T02 does not pre-apply gates to future folds, top-K groups, segments, leaves, or other not-yet-defined populations.

In [6]:
support_rows = []
for arm in (0, 1):
    n_arm = int(arm_counts[arm])
    support_rows.append({
        "finding_id": f"T02-GLOBAL-ARM-{arm}", "scope": "global_assignment_arm",
        "observed_condition": f"T={arm} has {n_arm} rows",
        "evidence_status": "INFO" if n_arm > 0 else "MATERIAL_CONCERN",
        "required_action": "PASS" if n_arm > 0 else "STOP",
        "implication": "Report the observed allocation; this does not prove randomization.",
    })
for row in treatment_outcome_summary.itertuples(index=False):
    support_rows.append({
        "finding_id": f"T02-GLOBAL-CELL-{row.treatment}-{row.conversion}",
        "scope": "global_joint_TY_cell",
        "observed_condition": f"T={row.treatment}, Y={row.conversion} has {row.n} rows ({row.within_arm_fraction:.8f} within arm)",
        "evidence_status": "INFO" if row.n > 0 else "WARNING",
        "required_action": "PASS" if row.n > 0 else "WARNING",
        "implication": "Downstream tasks must evaluate support again for their own realized folds, ranked groups, segments, or leaves.",
    })
for feature, rows in feature_arm_summary.groupby("feature", sort=True):
    overlap_exists = bool(rows["univariate_observed_range_overlap_diagnostic"].iloc[0])
    support_rows.append({
        "finding_id": f"T02-UNIVARIATE-RANGE-{feature}",
        "scope": "univariate_observed_range_overlap_diagnostic",
        "observed_condition": f"{feature}: observed arm ranges {'overlap' if overlap_exists else 'do not overlap'}",
        "evidence_status": "INFO" if overlap_exists else "WARNING",
        "required_action": "PASS" if overlap_exists else "WARNING",
        "implication": "Descriptive only; T03 performs formal design-calibrated diagnostics and this is not proof of multivariate overlap/positivity.",
    })
support_findings = pd.DataFrame(support_rows)
support_findings.insert(0, "run_id", RUN_ID)
support_findings.insert(1, "population", POPULATION)
write_csv_new(support_findings, RUN_ROOT, "tables/support_findings.csv")
display(support_findings)

,run_id,population,finding_id,scope,observed_condition,evidence_status,required_action,implication
0,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-GLOBAL-ARM-0,global_assignment_arm,T=0 has 2096937 rows,INFO,PASS,Report the observed allocation; this does not ...
1,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-GLOBAL-ARM-1,global_assignment_arm,T=1 has 11882655 rows,INFO,PASS,Report the observed allocation; this does not ...
2,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-GLOBAL-CELL-0-0,global_joint_TY_cell,"T=0, Y=0 has 2092874 rows (0.99806241 within arm)",INFO,PASS,Downstream tasks must evaluate support again f...
3,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-GLOBAL-CELL-0-1,global_joint_TY_cell,"T=0, Y=1 has 4063 rows (0.00193759 within arm)",INFO,PASS,Downstream tasks must evaluate support again f...
4,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-GLOBAL-CELL-1-0,global_joint_TY_cell,"T=1, Y=0 has 11845944 rows (0.99691054 within ...",INFO,PASS,Downstream tasks must evaluate support again f...
5,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-GLOBAL-CELL-1-1,global_joint_TY_cell,"T=1, Y=1 has 36711 rows (0.00308946 within arm)",INFO,PASS,Downstream tasks must evaluate support again f...
6,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-UNIVARIATE-RANGE-f0,univariate_observed_range_overlap_diagnostic,f0: observed arm ranges overlap,INFO,PASS,Descriptive only; T03 performs formal design-c...
7,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-UNIVARIATE-RANGE-f1,univariate_observed_range_overlap_diagnostic,f1: observed arm ranges overlap,INFO,PASS,Descriptive only; T03 performs formal design-c...
8,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-UNIVARIATE-RANGE-f10,univariate_observed_range_overlap_diagnostic,f10: observed arm ranges overlap,INFO,PASS,Descriptive only; T03 performs formal design-c...
9,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,T02-UNIVARIATE-RANGE-f11,univariate_observed_range_overlap_diagnostic,f11: observed arm ranges overlap,INFO,PASS,Descriptive only; T03 performs formal design-c...


## 6. Purposeful candidate figures

Figures are retained only when they add material pattern visibility beyond their source table. They are presentations derived from authoritative machine-readable tables, never independent evidence.

In [7]:
FIGURE_DIR = RUN_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "font.size": 9})
figure_catalog_rows = []

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
arm_table = treatment_outcome_summary.groupby(TREATMENT_COLUMN, as_index=False).agg(arm_n=("arm_n", "first"), treatment_fraction=("treatment_fraction", "first"))
conversion_table = treatment_outcome_summary.loc[treatment_outcome_summary[PRIMARY_OUTCOME] == 1, [TREATMENT_COLUMN, "within_arm_fraction"]]
axes[0].bar(["Control (T=0)", "Treated (T=1)"], arm_table["arm_n"], color=["#4C78A8", "#F58518"])
axes[0].set_title("Observed assignment-arm counts")
axes[0].set_ylabel("Released rows")
axes[0].set_ylim(0, float(arm_table["arm_n"].max()) * 1.18)
for i, row in arm_table.iterrows():
    axes[0].text(i, row.arm_n, f"{int(row.arm_n):,}\n({row.treatment_fraction:.2%})", ha="center", va="bottom")
axes[1].bar(["Control (T=0)", "Treated (T=1)"], conversion_table["within_arm_fraction"], color=["#4C78A8", "#F58518"])
axes[1].set_title("Observed conversion fraction within arm")
axes[1].set_ylabel("Conversion fraction")
axes[1].set_ylim(0, float(conversion_table["within_arm_fraction"].max()) * 1.18)
for i, value in enumerate(conversion_table["within_arm_fraction"]):
    axes[1].text(i, value, f"{value:.6%}", ha="center", va="bottom")
support_figure_rel = "figures/treatment_outcome_support.png"
save_figure_new(fig, RUN_ROOT, support_figure_rel, metadata={"Title": "T02 observed treatment and outcome support", "Software": f"matplotlib {matplotlib.__version__}"})
plt.close(fig)
figure_catalog_rows.append({
    "figure_id": "treatment_outcome_support", "figure": support_figure_rel, "retained": True,
    "source_identifier": "treatment_outcome_summary",
    "source_table": "tables/treatment_outcome_summary.csv",
    "denominator": f"Arm counts: all {row_count} validated unsplit rows; conversion fractions: rows within T=0 (n={int(arm_counts[0])}) or T=1 (n={int(arm_counts[1])}).",
    "question": "How do assignment-arm sizes and conversion rarity differ?",
    "observation": f"T=1 comprises {arm_counts[1] / row_count:.8f} of rows; observed conversion fractions are {float(conversion_table.loc[conversion_table[TREATMENT_COLUMN] == 0, 'within_arm_fraction'].iloc[0]):.8f} for T=0 and {float(conversion_table.loc[conversion_table[TREATMENT_COLUMN] == 1, 'within_arm_fraction'].iloc[0]):.8f} for T=1.",
    "interpretation": "Treatment-arm imbalance and outcome rarity are distinct descriptive properties and may create different downstream precision constraints.",
    "what_cannot_be_concluded": "The allocation does not prove randomization, and the raw arm-rate difference is not a causal effect or evidence of heterogeneous effects.",
    "implication": "Preserve exact T/Y cells; later estimator tasks must assess their own realized support without choosing a winner from this plot.",
    "material_information": "Juxtaposes two distinct imbalance scales that are easy to conflate in a row table.",
})

fig, axes = plt.subplots(3, 4, figsize=(15, 10), constrained_layout=True)
q_x = np.asarray(QUANTILES)
q_cols = [QUANTILE_COLUMNS[q] for q in QUANTILES]
for ax, feature in zip(axes.flat, FEATURE_COLUMNS):
    overall = feature_summary.loc[feature_summary["feature"] == feature, q_cols].iloc[0].to_numpy(dtype=float)
    arm0 = feature_arm_summary.loc[(feature_arm_summary["feature"] == feature) & (feature_arm_summary[TREATMENT_COLUMN] == 0), q_cols].iloc[0].to_numpy(dtype=float)
    arm1 = feature_arm_summary.loc[(feature_arm_summary["feature"] == feature) & (feature_arm_summary[TREATMENT_COLUMN] == 1), q_cols].iloc[0].to_numpy(dtype=float)
    ax.plot(q_x, overall, color="#777777", linestyle="--", marker="o", label="Overall")
    ax.plot(q_x, arm0, color="#4C78A8", marker="o", label="T=0")
    ax.plot(q_x, arm1, color="#F58518", marker="o", label="T=1")
    ax.set_title(feature)
    ax.set_xlabel("Quantile probability")
    ax.set_ylabel("Anonymous feature value")
axes.flat[0].legend()
quantile_figure_rel = "figures/feature_quantiles_overall_and_by_arm.png"
save_figure_new(fig, RUN_ROOT, quantile_figure_rel, metadata={"Title": "T02 feature quantiles overall and by assignment arm", "Software": f"matplotlib {matplotlib.__version__}"})
plt.close(fig)
figure_catalog_rows.append({
    "figure_id": "feature_quantiles_overall_and_by_arm", "figure": quantile_figure_rel, "retained": True,
    "source_identifier": "feature_summary_and_feature_arm_summary",
    "source_table": "tables/feature_summary.csv; tables/feature_arm_summary.csv",
    "denominator": f"Overall lines use non-null values among all {row_count} rows per feature; arm lines use non-null values within T=0 (n={int(arm_counts[0])}) and T=1 (n={int(arm_counts[1])}); exact non-null counts are in the source tables.",
    "question": "Do overall summaries conceal arm-specific centers or tails?",
    "observation": f"All {len(FEATURE_COLUMNS)} frozen features have overall, T=0, and T=1 quantile series on the declared probabilities; exact values and missing counts are recorded in the linked tables.",
    "interpretation": "Overall quantiles can conceal arm-specific distribution patterns, while univariate curves remain descriptive rather than a joint-support test.",
    "what_cannot_be_concluded": "These curves do not prove multivariate overlap or positivity and do not justify feature selection, causal meaning, or estimator superiority.",
    "implication": "Keep frozen X unchanged and carry these descriptive patterns into later task-specific diagnostics and implementation checks.",
    "material_information": "Small multiples expose center/tail shape and overall-versus-arm patterns across all frozen features without inventing semantics.",
})
figure_catalog = pd.DataFrame(figure_catalog_rows)
figure_catalog.insert(0, "run_id", RUN_ID)
figure_catalog.insert(1, "population", POPULATION)
write_csv_new(figure_catalog, RUN_ROOT, "tables/figure_catalog.csv")
display(figure_catalog)

,run_id,population,figure_id,figure,retained,source_identifier,source_table,denominator,question,observation,interpretation,what_cannot_be_concluded,implication,material_information
0,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,treatment_outcome_support,figures/treatment_outcome_support.png,True,treatment_outcome_summary,tables/treatment_outcome_summary.csv,Arm counts: all 13979592 validated unsplit row...,How do assignment-arm sizes and conversion rar...,T=1 comprises 0.85000013 of rows; observed con...,Treatment-arm imbalance and outcome rarity are...,"The allocation does not prove randomization, a...",Preserve exact T/Y cells; later estimator task...,Juxtaposes two distinct imbalance scales that ...
1,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_quantiles_overall_and_by_arm,figures/feature_quantiles_overall_and_by_arm.png,True,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Overall lines use non-null values among all 13...,Do overall summaries conceal arm-specific cent...,"All 12 frozen features have overall, T=0, and ...",Overall quantiles can conceal arm-specific dis...,These curves do not prove multivariate overlap...,Keep frozen X unchanged and carry these descri...,Small multiples expose center/tail shape and o...


## 7. Evidence-bounded interpretation and immutable closure

Every retained output follows: **Question → Observation → What cannot be concluded → Implication**.

In [8]:
control_rate = float(treatment_outcome_summary.loc[(treatment_outcome_summary[TREATMENT_COLUMN] == 0) & (treatment_outcome_summary[PRIMARY_OUTCOME] == 1), "within_arm_fraction"].iloc[0])
treated_rate = float(treatment_outcome_summary.loc[(treatment_outcome_summary[TREATMENT_COLUMN] == 1) & (treatment_outcome_summary[PRIMARY_OUTCOME] == 1), "within_arm_fraction"].iloc[0])
interpretation_rows = [{
    "output_id": "treatment_outcome_summary",
    "evidence_kind": "DESCRIPTIVE_TABLE",
    "source_identifier": "treatment_outcome_summary",
    "source_table": "tables/treatment_outcome_summary.csv",
    "denominator": f"All {row_count} validated unsplit rows for allocation; within-arm rows for conversion fractions.",
    "question": "How do observed treatment allocation and outcome rarity differ?",
    "observation": f"T=1 fraction is {arm_counts[1] / row_count:.8f}; observed conversion fractions are {control_rate:.8f} for T=0 and {treated_rate:.8f} for T=1.",
    "interpretation": "Treatment allocation imbalance and outcome rarity are separate descriptive properties with potentially different downstream precision consequences.",
    "what_cannot_be_concluded": "The allocation does not prove randomization, and the raw arm-rate difference alone does not establish a causal effect or heterogeneous effects.",
    "implication": "Carry exact global counts into T03; downstream tasks must verify support for their own realized populations and operations.",
}]
for row in feature_summary.itertuples(index=False):
    interpretation_rows.append({
        "output_id": f"feature_{row.feature}",
        "evidence_kind": "DESCRIPTIVE_TABLE",
        "source_identifier": "feature_summary_and_feature_arm_summary",
        "source_table": "tables/feature_summary.csv; tables/feature_arm_summary.csv",
        "denominator": f"Non-null {row.feature} values among all {int(row.n)} rows overall and within each observed treatment arm; exact counts are in the source tables.",
        "question": f"What are the observed distribution and cardinality of {row.feature}?",
        "observation": f"{row.feature}: mean={format_optional(row.mean)}, median={format_optional(row.q50)}, q01={format_optional(row.q01)}, q99={format_optional(row.q99)}, unique_non_null={int(row.n_unique_non_null)}. Arm-specific summaries are recorded separately.",
        "interpretation": "Overall and arm-specific summaries describe marginal location, spread, tails, and cardinality without assigning semantics.",
        "what_cannot_be_concluded": "No business meaning, causation, multivariate overlap/positivity, treatment-effect heterogeneity, or feature-selection decision follows from this description.",
        "implication": "Retain the feature in frozen X; use T03 and each downstream task's defined population for formal diagnostics and support gates.",
    })
estimator_observation = f"Observed arm counts are T=0: {int(arm_counts[0])} and T=1: {int(arm_counts[1])}; conversion counts are T=0: {int(joint_counts[1])} and T=1: {int(joint_counts[3])}; both global arms and all reported univariate ranges have observed support."
estimator_rows = [
    {
        "output_id": "bounded_implication_t_learner", "evidence_kind": "BOUNDED_ESTIMATOR_IMPLICATION",
        "source_identifier": "treatment_outcome_and_support_summaries", "source_table": "tables/treatment_outcome_summary.csv; tables/support_findings.csv",
        "denominator": f"All {row_count} validated unsplit rows; global T/Y cells and univariate observed ranges only.",
        "question": "How may the observed imbalance, rarity, and global support affect later T-Learner implementation?",
        "observation": estimator_observation,
        "interpretation": "Separate arm-specific outcome models may have different precision because their arm and conversion counts differ; rarity may increase outcome-model uncertainty.",
        "what_cannot_be_concluded": "T02 does not establish future fold support, comparative performance, a tuning rule, or T-Learner validity.",
        "implication": "Later implementation must report realized arm/event support and validation behavior without changing the frozen comparator role.",
    },
    {
        "output_id": "bounded_implication_x_learner", "evidence_kind": "BOUNDED_ESTIMATOR_IMPLICATION",
        "source_identifier": "treatment_outcome_and_support_summaries", "source_table": "tables/treatment_outcome_summary.csv; tables/support_findings.csv",
        "denominator": f"All {row_count} validated unsplit rows; global T/Y cells and univariate observed ranges only.",
        "question": "How may the observed imbalance, rarity, and global support affect later X-Learner implementation?",
        "observation": estimator_observation,
        "interpretation": "X-Learner's cross-arm imputation and combination steps make arm imbalance relevant, while rare outcomes may add uncertainty to nuisance predictions and imputed effects.",
        "what_cannot_be_concluded": "Imbalance does not prove X-Learner will outperform, and global/univariate support does not prove support in future folds or covariate neighborhoods.",
        "implication": "Later implementation must verify realized nuisance and pseudo-outcome behavior on development data without selecting a winner from T02.",
    },
    {
        "output_id": "bounded_implication_causal_forest", "evidence_kind": "BOUNDED_ESTIMATOR_IMPLICATION",
        "source_identifier": "treatment_outcome_and_support_summaries", "source_table": "tables/treatment_outcome_summary.csv; tables/support_findings.csv",
        "denominator": f"All {row_count} validated unsplit rows; global T/Y cells and univariate observed ranges only.",
        "question": "How may the observed imbalance, rarity, and global support affect later Causal Forest implementation?",
        "observation": estimator_observation,
        "interpretation": "Global support can coexist with sparse treated/control or outcome information in future local neighborhoods or leaves; rarity may constrain local precision.",
        "what_cannot_be_concluded": "T02 does not establish multivariate positivity, honesty/leaf support, implementation correctness, tuning thresholds, or Causal Forest performance.",
        "implication": "The later Causal Forest task must apply its declared local support and promotion checks to realized development structures.",
    },
]
interpretation_rows.extend(estimator_rows)
for row in figure_catalog.itertuples(index=False):
    interpretation_rows.append({
        "output_id": row.figure_id, "evidence_kind": "RETAINED_FIGURE",
        "source_identifier": row.source_identifier, "source_table": row.source_table,
        "denominator": row.denominator, "question": row.question,
        "observation": row.observation, "interpretation": row.interpretation,
        "what_cannot_be_concluded": row.what_cannot_be_concluded, "implication": row.implication,
    })
eda_interpretation = pd.DataFrame(interpretation_rows)
eda_interpretation.insert(0, "run_id", RUN_ID)
eda_interpretation.insert(1, "population", POPULATION)
write_csv_new(eda_interpretation, RUN_ROOT, "tables/eda_interpretation.csv")

resource_observations = {
    "run_id": RUN_ID, "population": POPULATION,
    "invariant_scan_elapsed_seconds": invariant_scan_elapsed_seconds,
    "peak_process_rss_bytes_observed": peak_rss_bytes,
    "feature_operations": feature_resource_rows,
    "operational_interpretation": "observations only; no universal RAM-percentage threshold",
}
write_json_new(RUN_ROOT, "audit/t02_resource_observations.json", resource_observations)

warning_count = int((support_findings["required_action"] == "WARNING").sum())
final_status = "COMPLETED_WARN" if warning_count else "COMPLETED_PASS"
t02_summary = {
    "run_id": RUN_ID, "status": final_status, "population": POPULATION,
    "rows": row_count, "features": list(FEATURE_COLUMNS),
    "warning_count": warning_count,
    "model_training": False, "outer_split_constructed": False,
    "held_out_evaluation_accessed": False, "feature_selection_performed": False,
    "limitations": [
        "EDA is descriptive and does not establish causal effects.",
        "Anonymous features receive no invented business semantics.",
        "Univariate observed-range overlap is not proof of multivariate overlap or positivity.",
        "Future folds, ranked groups, segments, and leaves require task-specific support checks when defined.",
        "T02 reports missingness only; model-facing handling is deferred to T04 and estimator-specific contracts.",
    ],
}
write_json_new(RUN_ROOT, "audit/t02_summary.json", t02_summary)

manifest_path = finalize_artifact_manifest(
    RUN_ROOT, run_id=RUN_ID, final_status=final_status, created_at_utc=utc_now(),
    stage=STAGE, population=POPULATION,
    external_artifacts=[
        {"path": data_manifest["processed_path"], "role": "manifest_selected_processed_derivative", "sha256": processed_sha256, "size_bytes": selector.processed_path.stat().st_size, "status": "PASS"},
        {"path": "notebooks/03_exploratory_data_analysis.ipynb#sources", "role": "human_readable_protocol_source", "sha256": notebook_source_hash, "status": "PASS"},
        {"path": "src/data.py", "role": "reusable_data_and_artifact_contract", "sha256": src_data_hash, "status": "PASS"},
    ],
)
artifact_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for artifact in artifact_manifest["artifacts"]:
    path = RUN_ROOT / artifact["path"]
    assert path.is_file() and path.stat().st_size == artifact["size_bytes"]
    assert sha256_file(path) == artifact["sha256"]
    assert artifact["stage"] == STAGE and artifact["population"] == POPULATION
listed_artifacts = {artifact["path"] for artifact in artifact_manifest["artifacts"]}
physical_files = {path.relative_to(RUN_ROOT).as_posix() for path in RUN_ROOT.rglob("*") if path.is_file()}
assert physical_files == listed_artifacts | {"audit/artifact_manifest.json"}

display(eda_interpretation)
display(Markdown(f"""### T02 implementation result: `{final_status}`

- Run: `{RUN_ID}`
- Rows characterized: `{row_count:,}`
- Authoritative root: `{RUN_ROOT.relative_to(REPO_ROOT).as_posix()}`
- Manifest: `{manifest_path.relative_to(REPO_ROOT).as_posix()}`
- Warnings: `{warning_count}`
- Model training / split construction / held-out evaluation: `False / False / False`
"""))

,run_id,population,output_id,evidence_kind,source_identifier,source_table,denominator,question,observation,interpretation,what_cannot_be_concluded,implication
0,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,treatment_outcome_summary,DESCRIPTIVE_TABLE,treatment_outcome_summary,tables/treatment_outcome_summary.csv,All 13979592 validated unsplit rows for alloca...,How do observed treatment allocation and outco...,T=1 fraction is 0.85000013; observed conversio...,Treatment allocation imbalance and outcome rar...,"The allocation does not prove randomization, a...",Carry exact global counts into T03; downstream...
1,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_f0,DESCRIPTIVE_TABLE,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Non-null f0 values among all 13979592 rows ove...,What are the observed distribution and cardina...,"f0: mean=19.620297, median=21.923413, q01=12.6...",Overall and arm-specific summaries describe ma...,"No business meaning, causation, multivariate o...",Retain the feature in frozen X; use T03 and ea...
2,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_f1,DESCRIPTIVE_TABLE,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Non-null f1 values among all 13979592 rows ove...,What are the observed distribution and cardina...,"f1: mean=10.069977, median=10.059654, q01=10.0...",Overall and arm-specific summaries describe ma...,"No business meaning, causation, multivariate o...",Retain the feature in frozen X; use T03 and ea...
3,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_f2,DESCRIPTIVE_TABLE,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Non-null f2 values among all 13979592 rows ove...,What are the observed distribution and cardina...,"f2: mean=8.4465823, median=8.2143828, q01=8.21...",Overall and arm-specific summaries describe ma...,"No business meaning, causation, multivariate o...",Retain the feature in frozen X; use T03 and ea...
4,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_f3,DESCRIPTIVE_TABLE,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Non-null f3 values among all 13979592 rows ove...,What are the observed distribution and cardina...,"f3: mean=4.1789231, median=4.6798816, q01=-1.7...",Overall and arm-specific summaries describe ma...,"No business meaning, causation, multivariate o...",Retain the feature in frozen X; use T03 and ea...
5,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_f4,DESCRIPTIVE_TABLE,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Non-null f4 values among all 13979592 rows ove...,What are the observed distribution and cardina...,"f4: mean=10.338837, median=10.280525, q01=10.2...",Overall and arm-specific summaries describe ma...,"No business meaning, causation, multivariate o...",Retain the feature in frozen X; use T03 and ea...
6,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_f5,DESCRIPTIVE_TABLE,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Non-null f5 values among all 13979592 rows ove...,What are the observed distribution and cardina...,"f5: mean=4.0285129, median=4.1154534, q01=2.23...",Overall and arm-specific summaries describe ma...,"No business meaning, causation, multivariate o...",Retain the feature in frozen X; use T03 and ea...
7,t02_eda_20260813T094750Z_546902,validated_unsplit_released_rows,feature_f6,DESCRIPTIVE_TABLE,feature_summary_and_feature_arm_summary,tables/feature_summary.csv; tables/feature_arm...,Non-null f6 values among all 13979592 rows ove...,What are the observed distribution and cardina...,"f6: mean=-4.1553555, median=-2.4111146, q01=-1...",Overall and arm-specific summaries describe ma...,"No business meaning, causation, multivariate

### T02 implementation result: `COMPLETED_PASS`

- Run: `t02_eda_20260813T094750Z_546902`
- Rows characterized: `13,979,592`
- Authoritative root: `outputs/runs/t02_eda_20260813T094750Z_546902`
- Manifest: `outputs/runs/t02_eda_20260813T094750Z_546902/audit/artifact_manifest.json`
- Warnings: `0`
- Model training / split construction / held-out evaluation: `False / False / False`
